<a href="https://colab.research.google.com/github/doantran1126/LLM/blob/main/kyc_copilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
"""
KYC Copilot — Final ver (Model mới + Robust Error Handling)
==========================================
Install: pip install gradio google-genai openpyxl pandas rapidfuzz
Run: python kyc_copilot.py
"""

import json
import mimetypes
import os
import re
import tempfile
import time
from datetime import datetime

import gradio as gr
import pandas as pd
from rapidfuzz import fuzz

try:
    from google import genai
    from google.genai import types as genai_types
except ImportError as e:
    raise ImportError("Missing google-genai.") from e

try:
    from dotenv import load_dotenv
    load_dotenv()
except ImportError:
    pass

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment

# ===== MODEL =====
DEFAULT_MODEL = "gemini-3.6-flash"
MODEL_PRESETS = [
    "gemini-3.6-flash",
    "gemini-3.5-flash",
    "gemini-3.5-flash-lite",
    "gemini-3.1-pro-preview",
]


def _resolve_model(model):
    m = (model or "").strip()
    return m or DEFAULT_MODEL


# =========================================================================
# SCHEMA
# =========================================================================

DOC_TYPES = ["ID_CARD", "KYC_FORM", "PROOF_OF_ADDRESS"]
DOC_LABELS = {
    "ID_CARD": "ID Card / Passport",
    "KYC_FORM": "KYC Form",
    "PROOF_OF_ADDRESS": "Proof of Address",
}
FIELD_LABELS = {
    "full_name": "Full Name", "dob": "Date of Birth", "id_number": "ID Number",
    "gender": "Gender", "address": "Address", "expiry_date": "Expiry Date",
}
REQUIRED_FIELDS_BY_DOC = {
    "ID_CARD": ["full_name", "dob", "id_number", "gender", "expiry_date"],
    "KYC_FORM": ["full_name", "dob", "id_number", "gender", "address"],
    "PROOF_OF_ADDRESS": ["full_name", "address"],
}
FIELD_DOC_MAP = {
    "full_name": ["ID_CARD", "KYC_FORM", "PROOF_OF_ADDRESS"],
    "dob": ["ID_CARD", "KYC_FORM"],
    "id_number": ["ID_CARD", "KYC_FORM"],
    "gender": ["ID_CARD", "KYC_FORM"],
    "address": ["KYC_FORM", "PROOF_OF_ADDRESS"],
    "expiry_date": ["ID_CARD"],
}
FIELD_MATCH_TYPE = {
    "full_name": "fuzzy", "address": "fuzzy",
    "id_number": "exact", "dob": "exact", "gender": "exact",
}
FUZZY_THRESHOLD = 85

DECISION_COLORS = {
    "ACCEPT": "#059669",
    "REQUEST_MORE_INFO": "#d97706",
    "MANUAL_REVIEW": "#dc2626",
    "SYSTEM_ERROR": "#7c3aed",
}
DECISION_ICONS = {"ACCEPT": "✓", "REQUEST_MORE_INFO": "!",
                   "MANUAL_REVIEW": "⚠", "SYSTEM_ERROR": "×"}

API_CALLS = {"extract": 0, "summary": 0, "chatbot": 0}
VALID_IMAGE_EXT = (".png", ".jpg", ".jpeg", ".webp", ".gif", ".bmp")


def _is_valid_image(path) -> bool:
    if not path:
        return True
    if isinstance(path, str):
        return path.lower().endswith(VALID_IMAGE_EXT)
    if isinstance(path, dict):
        name = path.get("name", "") or path.get("path", "")
        return str(name).lower().endswith(VALID_IMAGE_EXT)
    return False


# =========================================================================
# CONFIDENCE
# =========================================================================

DATE_REGEX = re.compile(r"^\d{4}-\d{2}-\d{2}$")
ID_NUMBER_REGEX = re.compile(r"^(\d{9}|\d{12}|[A-Za-z]\d{7,8})$")
NAME_REGEX = re.compile(r"^[A-Za-zÀ-ỹ\s]{3,80}$")
GENDER_VALUES = {"male", "female", "nam", "nu", "nữ"}


def calculate_field_confidence(field: str, value) -> str:
    if value is None or str(value).strip() == "":
        return "VERY_LOW"
    v = str(value).strip()
    if field in ("dob", "expiry_date"):
        if not DATE_REGEX.match(v):
            return "LOW"
        try:
            datetime.strptime(v, "%Y-%m-%d")
            return "HIGH"
        except ValueError:
            return "LOW"
    if field == "id_number":
        cleaned = v.replace(" ", "").replace("-", "")
        if ID_NUMBER_REGEX.match(cleaned):
            return "HIGH"
        if 6 <= len(cleaned) <= 15:
            return "LOW"
        return "VERY_LOW"
    if field == "full_name":
        if NAME_REGEX.match(v) and len(v.split()) >= 2:
            return "HIGH"
        if any(ch.isdigit() for ch in v):
            return "LOW"
        return "MEDIUM"
    if field == "gender":
        return "HIGH" if v.lower() in GENDER_VALUES else "LOW"
    if field == "address":
        if len(v) >= 10 and any(ch.isdigit() for ch in v):
            return "HIGH"
        if len(v) >= 5:
            return "MEDIUM"
        return "LOW"
    return "MEDIUM"


# =========================================================================
# GEMINI
# =========================================================================

def _guess_mime_type(path: str) -> str:
    mime, _ = mimetypes.guess_type(path)
    return mime or "image/jpeg"


def _get_client(api_key: str):
    key = (api_key or "").strip() or os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        raise ValueError("Missing Gemini API Key.")
    return genai.Client(api_key=key)


def _is_retryable_error(err_str: str) -> bool:
    return any(x in err_str for x in [
        "503", "429", "UNAVAILABLE", "RESOURCE_EXHAUSTED", "overloaded", "quota",
    ])


def _extract_retry_delay(err_str: str, default: int = 12) -> int:
    m = re.search(r"retryDelay['\"]?\s*:\s*['\"]?(\d+)", err_str)
    if m:
        return max(int(m.group(1)), 10)
    return default


def _parse_json_response(text: str):
    """Robustly parse JSON from model response, stripping markdown fences."""
    if not text:
        raise ValueError("Empty response from model")
    s = str(text).strip()
    s = re.sub(r"^```(?:json|JSON)?\s*", "", s)
    s = re.sub(r"\s*```\s*$", "", s)
    s = s.strip()
    try:
        return json.loads(s)
    except json.JSONDecodeError:
        pass
    m = re.search(r"\{[\s\S]*\}", s)
    if m:
        try:
            return json.loads(m.group(0))
        except json.JSONDecodeError as e:
            raise ValueError(f"JSON parse failed: {e}. Raw: {s[:300]}")
    raise ValueError(f"No JSON object found. Raw response: {s[:300]}")


def test_gemini_api(api_key: str, model: str) -> str:
    """Quick test that API key + model are valid and responsive."""
    model = _resolve_model(model)
    key = (api_key or "").strip() or os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        return "❌ Chưa nhập API Key"
    try:
        client = _get_client(key)
        response = client.models.generate_content(
            model=model,
            contents="Reply with exactly one word: OK",
            config=genai_types.GenerateContentConfig(temperature=0),
        )
        text = (response.text or "").strip()
        return f"✅ **{model}** hoạt động tốt! Response: `{text[:80]}`"
    except Exception as e:
        err = str(e)
        if "404" in err or "not found" in err.lower():
            return (f"❌ **Model `{model}` không tồn tại** hoặc không có quyền truy cập.\n"
                    f"→ Hãy chọn 1 trong: `gemini-3.6-flash`, `gemini-3.5-flash`, "
                    f"`gemini-3.5-flash-lite`, `gemini-3.1-pro-preview`.\n\n"
                    f"Chi tiết: `{err[:200]}`")
        if "API_KEY" in err.upper() or "401" in err or "403" in err:
            return f"❌ **API Key không hợp lệ**: `{err[:200]}`"
        if "429" in err or "quota" in err.lower():
            return f"⚠️ **Hết quota**: `{err[:200]}`"
        return f"❌ **Lỗi khác** với `{model}`: `{err[:300]}`"


def call_gemini_extraction(api_key: str, image_path: str, doc_type: str,
                           model: str = DEFAULT_MODEL, max_retries: int = 3) -> dict:
    model = _resolve_model(model)
    required_fields = REQUIRED_FIELDS_BY_DOC.get(doc_type, [])
    field_list_str = ", ".join(required_fields)
    prompt = f"""You are a document OCR extraction engine for a KYC system.
Document type: {DOC_LABELS.get(doc_type, doc_type)}.

Look carefully at the image and extract these exact fields: {field_list_str}.

Return ONLY a valid JSON object with exactly these keys. Example shape:
{{
  "full_name": "Nguyen Van An",
  "dob": "1998-03-15",
  "id_number": "012345678901",
  "gender": "Male",
  "address": "123 Nguyen Trai, District 1, HCMC",
  "expiry_date": "2030-03-15"
}}

Rules:
- Return ALL keys listed above. Use empty string "" if a field is not visible.
- Dates MUST be YYYY-MM-DD format. Convert from any other format if needed.
- Do NOT wrap output in markdown code fences.
- Do NOT add any explanation text. JSON only."""

    RETRY_WAITS = [10, 20, 30]
    last_error = None
    raw_response = None

    for attempt in range(max_retries):
        try:
            with open(image_path, "rb") as f:
                image_bytes = f.read()
            client = _get_client(api_key)
            response = client.models.generate_content(
                model=model,
                contents=[
                    genai_types.Part.from_bytes(data=image_bytes,
                                                  mime_type=_guess_mime_type(image_path)),
                    prompt,
                ],
                config=genai_types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0,
                ),
            )
            raw_response = (response.text or "").strip()
            data = _parse_json_response(raw_response)
            if not isinstance(data, dict):
                raise ValueError(f"Model returned {type(data).__name__}, expected dict")

            result = {}
            for k in required_fields:
                v = data.get(k, "")
                if v is None:
                    v = ""
                result[k] = str(v).strip() if v != "" else ""
            result["_error"] = None
            result["_raw"] = raw_response[:500]
            API_CALLS["extract"] += 1
            return result
        except Exception as e:
            error_str = str(e)
            last_error = error_str
            print(f"[EXTRACT FAIL] {doc_type} attempt {attempt+1}/{max_retries}: {error_str[:300]}")
            if raw_response:
                print(f"  Raw response (first 300 chars): {raw_response[:300]}")
            if _is_retryable_error(error_str) and attempt < max_retries - 1:
                wait = max(RETRY_WAITS[attempt] if attempt < len(RETRY_WAITS) else 15,
                           _extract_retry_delay(error_str))
                print(f"  → Retrying in {wait}s...")
                time.sleep(wait)
                continue
            break

    data = {f: "" for f in required_fields}
    data["_error"] = f"Failed after {max_retries} attempts: {last_error}"
    data["_raw"] = (raw_response or "")[:500]
    return data


def build_documents_from_uploads(api_key, id_path, kyc_path, poa_path,
                                 model: str = DEFAULT_MODEL, progress=None):
    model = _resolve_model(model)
    uploads = {"ID_CARD": id_path, "KYC_FORM": kyc_path, "PROOF_OF_ADDRESS": poa_path}
    documents, errors = {}, []
    steps = [d for d, p in uploads.items() if p]
    for i, doc_type in enumerate(steps):
        if progress is not None:
            progress(0.10 + 0.15 * (i + 1),
                     desc=f"📄 Reading {DOC_LABELS[doc_type]} ({model})...")
        raw = call_gemini_extraction(api_key, uploads[doc_type], doc_type, model=model)
        err = raw.pop("_error", None)
        raw.pop("_raw", None)
        if err:
            errors.append(f"{DOC_LABELS[doc_type]}: {err}")
        extracted = {}
        for field, value in raw.items():
            extracted[field] = {
                "value": value,
                "confidence": calculate_field_confidence(field, value),
            }
        documents[doc_type] = {
            "doc_type": doc_type,
            "file_name": os.path.basename(uploads[doc_type]),
            "extracted_fields": extracted,
        }
    return documents, errors


def _build_offline_summary(decision, risk_flags, completeness) -> str:
    parts = []
    if decision == "ACCEPT":
        parts.append(
            "**Case verified successfully.** All three required documents "
            "(ID Card, KYC Form, Proof of Address) are complete and consistent. "
            "The deterministic Rule Engine validated all extracted fields, "
            "confirmed cross-document consistency, and found no risk indicators. "
            "The applicant meets all onboarding requirements for automatic approval."
        )
    elif decision == "REQUEST_MORE_INFO":
        missing = []
        if completeness["missing_documents"]:
            labels = [DOC_LABELS.get(d, d) for d in completeness["missing_documents"]]
            missing.append(f"missing document{'s' if len(labels) > 1 else ''}: {', '.join(labels)}")
        if completeness["missing_fields"]:
            n = len(completeness["missing_fields"])
            missing.append(f"{n} incomplete field{'s' if n > 1 else ''}")
        reason = " and ".join(missing) if missing else "incomplete submission"
        parts.append(
            f"**Additional information required.** The submitted package is incomplete — "
            f"{reason}. The Rule Engine cannot proceed with verification until the applicant "
            f"provides the missing items."
        )
    elif decision == "MANUAL_REVIEW":
        flag_msgs = "; ".join(f["message"] for f in risk_flags)
        n = len(risk_flags)
        parts.append(
            f"**Manual review required.** The Rule Engine detected "
            f"{n} risk flag{'s' if n > 1 else ''} that prevent automatic approval. "
            f"Key findings: {flag_msgs}. A compliance reviewer must inspect the evidence."
        )
    elif decision == "SYSTEM_ERROR":
        parts.append(
            "**System error.** The AI extraction step failed for all uploaded documents. "
            "No fields could be read. Please check:\n"
            "1. API Key is valid (use the **Test API** button)\n"
            "2. Model name is correct\n"
            "3. Images are readable (clear scans, not blurry)\n"
            "4. Quota is not exceeded"
        )
    parts.append("_This is a recommendation only. Final decision requires human approval._")
    return "\n\n".join(parts)


def generate_llm_summary_via_gemini(api_key, decision, risk_flags, completeness,
                                    model: str = DEFAULT_MODEL, max_retries: int = 1):
    model = _resolve_model(model)
    facts = {
        "decision": decision, "risk_flags": risk_flags,
        "missing_documents": completeness["missing_documents"],
        "missing_fields": completeness["missing_fields"],
    }
    prompt = f"""You are a KYC compliance assistant. The decision below was made by a
deterministic Rule Engine — do NOT change it. Write a 3-5 sentence summary explaining WHY.
End with exactly: "This is a recommendation only. Final decision requires human approval."

FACTS:
{json.dumps(facts, ensure_ascii=False)}"""

    for attempt in range(max_retries):
        try:
            client = _get_client(api_key)
            response = client.models.generate_content(
                model=model, contents=prompt,
                config=genai_types.GenerateContentConfig(temperature=0.2),
            )
            API_CALLS["summary"] += 1
            return {"text": (response.text or "").strip(), "used_fallback": False}
        except Exception as e:
            if _is_retryable_error(str(e)) and attempt < max_retries - 1:
                time.sleep(_extract_retry_delay(str(e)))
                continue
            break

    return {"text": _build_offline_summary(decision, risk_flags, completeness),
            "used_fallback": False}


def build_case_context_text(result: dict) -> str:
    lines = [f"Case ID: {result['case_id']}", f"Decision: {result['decision']}"]
    lines.append("Fields:")
    for doc_type, doc in result["documents"].items():
        for field, info in doc["extracted_fields"].items():
            lines.append(f"  {doc_type}.{field} = '{info['value']}' ({info['confidence']})")
    lines.append("Cross-check:")
    for item in result["cross_check_results"]:
        lines.append(f"  {item['field']}: {item['status']}")
    lines.append("Risk flags:")
    for flag in result["risk_flags"]:
        lines.append(f"  {flag['type']}: {flag['message']}")
    if not result["risk_flags"]:
        lines.append("  (none)")
    return "\n".join(lines)


def ask_ai_assistant(api_key, question, current_result, chat_history,
                     model: str = DEFAULT_MODEL):
    if not question or not question.strip():
        return chat_history, ""

    model = _resolve_model(model)

    def _append(history, q, a):
        history = list(history or [])
        return history + [{"role": "user", "content": q}, {"role": "assistant", "content": a}]

    if current_result is None:
        return _append(chat_history, question,
                       "No case open. Please upload documents and click Activate first."), ""
    if not ((api_key or "").strip() or os.getenv("GEMINI_API_KEY", "").strip()):
        return _append(chat_history, question, "Please enter a Gemini API Key."), ""

    prompt = f"""You are a KYC assistant. Use ONLY this case data. Do NOT state a final decision.

CASE:
{build_case_context_text(current_result)}

Question: {question}"""

    try:
        client = _get_client(api_key)
        response = client.models.generate_content(
            model=model, contents=prompt,
            config=genai_types.GenerateContentConfig(temperature=0.2),
        )
        answer = (response.text or "").strip()
        API_CALLS["chatbot"] += 1
    except Exception as e:
        err = str(e)
        if "429" in err or "quota" in err.lower():
            answer = ("I'm currently operating in offline mode due to AI quota limits. "
                      "The decision for this case was computed entirely by the deterministic "
                      "Rule Engine and remains accurate.")
        else:
            answer = f"AI service temporarily unavailable (model: {model}). Rule Engine decision unaffected."

    return _append(chat_history, question, answer), ""


# =========================================================================
# RULE ENGINE
# =========================================================================

def strip_accents(text: str) -> str:
    import unicodedata
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.replace("đ", "d").replace("Đ", "D")


def normalize_text(value) -> str:
    if value is None:
        return ""
    return re.sub(r"\s+", " ", strip_accents(str(value).strip().lower()))


def match_field(field, val_a, val_b):
    na, nb = normalize_text(val_a), normalize_text(val_b)
    mt = FIELD_MATCH_TYPE.get(field, "exact")
    if mt == "fuzzy":
        return "MATCH" if fuzz.ratio(na, nb) >= FUZZY_THRESHOLD else "MISMATCH"
    return "MATCH" if na == nb else "MISMATCH"


def cross_check(documents: dict):
    results = []
    for field, doc_list in FIELD_DOC_MAP.items():
        present = [d for d in doc_list if d in documents and field in documents[d]["extracted_fields"]]
        if len(present) < 2:
            continue
        comparisons, values = [], {}
        for d in present:
            val = documents[d]["extracted_fields"][field]["value"]
            comparisons.append({"document": d, "value": val})
            values[d] = val
        overall = "MATCH"
        for i in range(len(present)):
            for j in range(i + 1, len(present)):
                if match_field(field, values[present[i]], values[present[j]]) == "MISMATCH":
                    overall = "MISMATCH"
        results.append({"field": field, "status": overall,
                        "match_type": FIELD_MATCH_TYPE.get(field, "exact"),
                        "comparisons": comparisons})
    return results


def completeness_check(documents: dict):
    missing_docs = [d for d in DOC_TYPES if d not in documents]
    missing_fields = []
    for d, doc in documents.items():
        for field in REQUIRED_FIELDS_BY_DOC.get(d, []):
            if not doc["extracted_fields"].get(field, {}).get("value"):
                missing_fields.append(f"{d}.{field}")
    return {"missing_documents": missing_docs, "missing_fields": missing_fields}


def expiry_check(documents: dict, today=None):
    if today is None:
        today = datetime.now().date()
    id_doc = documents.get("ID_CARD")
    if not id_doc:
        return {"expired": None}
    expiry_val = id_doc["extracted_fields"].get("expiry_date", {}).get("value")
    if not expiry_val:
        return {"expired": None}
    try:
        expiry_date = datetime.strptime(expiry_val, "%Y-%m-%d").date()
    except ValueError:
        return {"expired": None}
    return {"expired": expiry_date < today}


def confidence_check(documents: dict):
    low = []
    for d, doc in documents.items():
        for field, info in doc["extracted_fields"].items():
            if info.get("confidence") in ("LOW", "VERY_LOW"):
                low.append(f"{d}.{field}")
    return {"overall": "LOW" if low else "HIGH", "low_confidence_fields": low}


def build_risk_flags_and_evidence(documents, cc_results, expiry_result, confidence_result):
    risk_flags, evidence = [], []
    for item in cc_results:
        if item["status"] == "MISMATCH":
            ft = f"{item['field'].upper()}_MISMATCH"
            risk_flags.append({"type": ft, "severity": "HIGH",
                                "message": f"{FIELD_LABELS.get(item['field'], item['field'])} does not match"})
            for comp in item["comparisons"]:
                evidence.append({"risk_flag": ft, "document": comp["document"],
                                  "field": item["field"], "value": comp["value"]})
    if expiry_result.get("expired"):
        risk_flags.append({"type": "EXPIRED_DOCUMENT", "severity": "HIGH",
                            "message": "ID document expired"})
        ev = documents["ID_CARD"]["extracted_fields"]["expiry_date"]["value"]
        evidence.append({"risk_flag": "EXPIRED_DOCUMENT", "document": "ID_CARD",
                          "field": "expiry_date", "value": ev})
    if confidence_result["overall"] == "LOW":
        for lf in confidence_result["low_confidence_fields"]:
            doc, field = lf.split(".")
            risk_flags.append({"type": "LOW_CONFIDENCE_FIELD", "severity": "MEDIUM",
                                "message": f"Low confidence: {FIELD_LABELS.get(field, field)} in {DOC_LABELS.get(doc, doc)}"})
            evidence.append({"risk_flag": "LOW_CONFIDENCE_FIELD", "document": doc,
                              "field": field, "value": documents[doc]["extracted_fields"][field]["value"]})
    return risk_flags, evidence


def decision_engine(completeness, cc_results, expiry_result, confidence_result,
                    extraction_errors=None):
    if extraction_errors:
        total_required = sum(len(REQUIRED_FIELDS_BY_DOC.get(d, []))
                              for d in DOC_TYPES)
        total_empty = len(completeness["missing_fields"])
        if total_empty >= total_required:
            return "SYSTEM_ERROR"

    if completeness["missing_documents"] or completeness["missing_fields"]:
        return "REQUEST_MORE_INFO"
    if any(item["status"] == "MISMATCH" for item in cc_results):
        return "MANUAL_REVIEW"
    if expiry_result.get("expired"):
        return "MANUAL_REVIEW"
    if confidence_result["overall"] == "LOW":
        return "MANUAL_REVIEW"
    return "ACCEPT"


# =========================================================================
# UI BUILDERS
# =========================================================================

def build_sidebar_html(history: list, current_result: dict = None) -> str:
    total = len(history)
    accepted = sum(1 for r in history if r["decision"] == "ACCEPT")
    review = sum(1 for r in history if r["decision"] == "MANUAL_REVIEW")
    info = sum(1 for r in history if r["decision"] == "REQUEST_MORE_INFO")
    err_count = sum(1 for r in history if r["decision"] == "SYSTEM_ERROR")

    total_api = API_CALLS["extract"] + API_CALLS["summary"] + API_CALLS["chatbot"]
    current_model = (current_result or {}).get("model") or DEFAULT_MODEL

    case_items = ""
    if not history:
        case_items = '<div style="padding:20px 0; text-align:center; color:#94a3b8; font-size:11px;">No cases in this session</div>'
    else:
        for r in reversed(history[-6:]):
            dec = r["decision"]
            color = DECISION_COLORS.get(dec, "#64748b")
            is_current = current_result and r["case_id"] == current_result.get("case_id")
            bg = "#f1f5f9" if is_current else "transparent"
            created = r.get("created_at", "")
            time_str = ""
            if created:
                try:
                    dt = datetime.fromisoformat(created)
                    time_str = dt.strftime("%H:%M:%S")
                except Exception:
                    time_str = created[:19].replace("T", " ")

            case_items += f'''
            <div style="padding:8px 10px; border-radius:6px; background:{bg};
                        margin-bottom:4px; cursor:pointer; transition:background 0.1s;"
                 onmouseover="this.style.background='#f8fafc'"
                 onmouseout="this.style.background='{bg}'">
                <div style="display:flex; align-items:center; justify-content:space-between; gap:8px;">
                    <div style="display:flex; align-items:center; gap:8px; min-width:0; flex:1;">
                        <span style="width:5px; height:5px; border-radius:50%; background:{color}; flex-shrink:0;"></span>
                        <span style="font-size:11px; font-weight:500; color:#334155;
                                    white-space:nowrap; overflow:hidden; text-overflow:ellipsis;
                                    font-family:'JetBrains Mono', monospace; letter-spacing:-0.3px;">
                            {r["case_id"][-10:]}
                        </span>
                    </div>
                    <span style="font-size:9px; font-weight:600; color:{color};
                                letter-spacing:0.5px;">
                        {dec.split("_")[0]}
                    </span>
                </div>
                <div style="font-size:9px; color:#94a3b8; padding-left:13px; margin-top:2px;
                            font-family:'JetBrains Mono', monospace;">
                    {time_str}
                </div>
            </div>
            '''

    return f'''
    <div id="kyc-sidebar" style="display:flex; flex-direction:column; height:100%; width:100%;
                background:#ffffff; overflow:hidden;">

        <div style="padding:20px 20px 18px; border-bottom:1px solid #e5e7eb; flex-shrink:0;">
            <div style="display:flex; align-items:center; gap:11px;">
                <div style="width:32px; height:32px; border-radius:8px;
                            background:#0f172a;
                            display:flex; align-items:center; justify-content:center;
                            color:#ffffff; font-size:15px; font-weight:800;
                            letter-spacing:-0.5px;">K</div>
                <div>
                    <div style="font-size:13px; font-weight:600; color:#0f172a;
                                letter-spacing:-0.2px; line-height:1.1;">
                        KYC Copilot
                    </div>
                    <div style="font-size:9px; color:#94a3b8; letter-spacing:1px;
                                font-weight:600; margin-top:2px; text-transform:uppercase;">
                        Enterprise Suite
                    </div>
                </div>
            </div>
        </div>

        <div style="padding:16px 20px 14px; border-bottom:1px solid #e5e7eb; flex-shrink:0;">
            <div style="font-size:9px; font-weight:700; color:#94a3b8;
                        letter-spacing:1.2px; text-transform:uppercase; margin-bottom:10px;">
                Session Overview
            </div>
            <div style="display:grid; grid-template-columns:1fr 1fr; gap:6px;">
                <div style="padding:9px 10px; background:#f8fafc;
                            border:1px solid #e5e7eb; border-radius:6px;">
                    <div style="font-size:19px; font-weight:700; color:#0f172a;
                                letter-spacing:-0.5px; line-height:1; font-variant-numeric:tabular-nums;">
                        {total}
                    </div>
                    <div style="font-size:9px; color:#94a3b8; font-weight:600;
                                letter-spacing:0.8px; margin-top:5px; text-transform:uppercase;">
                        Total
                    </div>
                </div>
                <div style="padding:9px 10px; background:#ecfdf5;
                            border:1px solid #a7f3d0; border-radius:6px;">
                    <div style="font-size:19px; font-weight:700; color:#059669;
                                letter-spacing:-0.5px; line-height:1; font-variant-numeric:tabular-nums;">
                        {accepted}
                    </div>
                    <div style="font-size:9px; color:#059669; font-weight:600;
                                letter-spacing:0.8px; margin-top:5px; text-transform:uppercase;">
                        Accept
                    </div>
                </div>
                <div style="padding:9px 10px; background:#fef2f2;
                            border:1px solid #fecaca; border-radius:6px;">
                    <div style="font-size:19px; font-weight:700; color:#dc2626;
                                letter-spacing:-0.5px; line-height:1; font-variant-numeric:tabular-nums;">
                        {review}
                    </div>
                    <div style="font-size:9px; color:#dc2626; font-weight:600;
                                letter-spacing:0.8px; margin-top:5px; text-transform:uppercase;">
                        Review
                    </div>
                </div>
                <div style="padding:9px 10px; background:#fffbeb;
                            border:1px solid #fde68a; border-radius:6px;">
                    <div style="font-size:19px; font-weight:700; color:#d97706;
                                letter-spacing:-0.5px; line-height:1; font-variant-numeric:tabular-nums;">
                        {info}
                    </div>
                    <div style="font-size:9px; color:#d97706; font-weight:600;
                                letter-spacing:0.8px; margin-top:5px; text-transform:uppercase;">
                        Info
                    </div>
                </div>
            </div>
        </div>

        <div style="flex:1 1 auto; padding:16px 20px; overflow-y:auto; min-height:0;">
            <div style="display:flex; align-items:center; justify-content:space-between; margin-bottom:10px;">
                <span style="font-size:9px; font-weight:700; color:#94a3b8;
                             letter-spacing:1.2px; text-transform:uppercase;">
                    Recent Cases
                </span>
                <span style="font-size:9px; color:#cbd5e1; font-weight:600;
                             font-family:'JetBrains Mono', monospace;">
                    {len(history)}
                </span>
            </div>
            {case_items}
        </div>

        <div style="padding:14px 20px; border-top:1px solid #e5e7eb; background:#fafbfc; flex-shrink:0;">
            <div style="display:flex; align-items:center; gap:7px; margin-bottom:8px;">
                <span style="width:6px; height:6px; border-radius:50%; background:#059669;"></span>
                <span style="font-size:10px; font-weight:600; color:#334155;
                             letter-spacing:0.5px;">
                    Operational
                </span>
            </div>
            <div style="font-size:10px; color:#94a3b8; line-height:1.6;">
                <div style="font-family:'JetBrains Mono', monospace;
                            word-break:break-all; color:#475569; font-weight:600;">
                    {current_model}
                </div>
                <div style="margin-top:2px;">Rule Engine v1.0</div>
                <div style="margin-top:6px; padding-top:6px; border-top:1px solid #e5e7eb;
                            display:flex; justify-content:space-between;">
                    <span style="font-size:9px; color:#94a3b8;">API calls</span>
                    <span style="font-size:9px; color:#475569; font-weight:700;
                                 font-family:'JetBrains Mono', monospace;">
                        {total_api}
                    </span>
                </div>
                <div style="display:flex; justify-content:space-between; margin-top:2px;">
                    <span style="font-size:9px; color:#cbd5e1;">
                        Ext / Sum / Chat
                    </span>
                    <span style="font-size:9px; color:#cbd5e1;
                                 font-family:'JetBrains Mono', monospace;">
                        {API_CALLS["extract"]} / {API_CALLS["summary"]} / {API_CALLS["chatbot"]}
                    </span>
                </div>
            </div>
        </div>

        <div id="reset-session-btn" role="button" tabindex="0" class="kyc-reset-html"
             onclick="var b=document.querySelector('#hidden-reset-btn button')||document.querySelector('#hidden-reset-btn');if(b)b.click();">
            ↻  Reset Session
        </div>
    </div>
    '''


def build_topbar_html(current_result: dict = None) -> str:
    case_info = ""
    if current_result:
        dec = current_result["decision"]
        color = DECISION_COLORS.get(dec, "#64748b")
        model_str = current_result.get("model", DEFAULT_MODEL)
        case_info = f'''
        <span style="color:#cbd5e1; margin:0 8px;">/</span>
        <span style="font-size:12px; color:#334155; font-weight:500;
                     font-family:'JetBrains Mono', monospace; letter-spacing:-0.3px;">
            {current_result["case_id"][-13:]}
        </span>
        <span style="display:inline-flex; align-items:center; gap:4px; padding:3px 9px;
                     border-radius:4px; background:{color}10; color:{color};
                     font-size:10px; font-weight:700; letter-spacing:0.5px;
                     margin-left:10px;">
            {DECISION_ICONS.get(dec, "?")} {dec.replace("_", " ")}
        </span>
        <span style="display:inline-flex; align-items:center; gap:4px; padding:3px 9px;
                     border-radius:4px; background:#f1f5f9; color:#475569;
                     font-size:10px; font-weight:600; letter-spacing:0.3px;
                     margin-left:6px; font-family:'JetBrains Mono', monospace;">
            {model_str}
        </span>
        '''

    return f'''
    <div style="display:flex; align-items:center; justify-content:space-between;
                padding:14px 32px; background:#ffffff;
                border-bottom:1px solid #e5e7eb;
                position:sticky; top:0; z-index:10;">
        <div style="display:flex; align-items:center;">
            <span style="font-size:13px; font-weight:600; color:#0f172a;
                         letter-spacing:-0.2px;">
                Onboarding Console
            </span>
            {case_info}
        </div>
        <div style="display:flex; align-items:center; gap:16px;">
            <span style="font-size:11px; color:#64748b; font-weight:500;
                         font-family:'JetBrains Mono', monospace;">
                {datetime.now().strftime("%H:%M · %d %b %Y").upper()}
            </span>
        </div>
    </div>
    '''


def build_decision_banner(decision, reason_count, processing_time=None) -> str:
    if decision not in DECISION_COLORS:
        return '''
        <div style="background:#ffffff; border:1px solid #e5e7eb; border-radius:10px;
                    padding:36px 28px; text-align:center;">
            <div style="font-size:10px; color:#94a3b8; letter-spacing:2px;
                        font-weight:700; text-transform:uppercase; margin-bottom:10px;">
                Awaiting Analysis
            </div>
            <div style="font-size:12px; color:#64748b;">
                Upload documents and click Activate to begin
            </div>
        </div>
        '''

    color = DECISION_COLORS.get(decision, "#64748b")
    icon = DECISION_ICONS.get(decision, "?")
    label = decision.replace("_", " ").title()
    if decision == "SYSTEM_ERROR":
        sub = "AI extraction failed — see error banner below"
    elif reason_count == 0:
        sub = "No issues detected — ready for approval"
    else:
        sub = f"{reason_count} issue{'s' if reason_count > 1 else ''} require attention"
    time_str = f'{processing_time}s' if processing_time else "—"

    return f'''
    <div style="background:#ffffff; border:1px solid #e5e7eb; border-radius:10px;
                padding:22px 26px; display:flex; align-items:center; gap:22px;">
        <div style="width:56px; height:56px; border-radius:10px;
                    background:{color}10; display:flex; align-items:center;
                    justify-content:center; font-size:26px; font-weight:700;
                    color:{color}; flex-shrink:0; border:1px solid {color}20;">
            {icon}
        </div>
        <div style="flex:1; min-width:0;">
            <div style="font-size:10px; color:#64748b; letter-spacing:1.5px;
                        font-weight:700; text-transform:uppercase; margin-bottom:5px;">
                Decision
            </div>
            <div style="font-size:22px; font-weight:700; color:#0f172a;
                        letter-spacing:-0.5px; line-height:1.15;">
                {label}
            </div>
            <div style="font-size:12px; color:#64748b; margin-top:4px;">
                {sub}
            </div>
        </div>
        <div style="display:flex; gap:20px; align-items:center; flex-shrink:0;">
            <div style="text-align:right;">
                <div style="font-size:9px; color:#94a3b8; letter-spacing:1px;
                            font-weight:700; text-transform:uppercase; margin-bottom:3px;">
                    Engine
                </div>
                <div style="font-size:11px; font-weight:600; color:#334155;
                            font-family:'JetBrains Mono', monospace;">
                    RULE-BASED
                </div>
            </div>
            <div style="width:1px; height:32px; background:#e5e7eb;"></div>
            <div style="text-align:right;">
                <div style="font-size:9px; color:#94a3b8; letter-spacing:1px;
                            font-weight:700; text-transform:uppercase; margin-bottom:3px;">
                    Processed
                </div>
                <div style="font-size:11px; font-weight:600; color:#334155;
                            font-family:'JetBrains Mono', monospace;">
                    {time_str}
                </div>
            </div>
        </div>
    </div>
    '''


def build_field_cards(documents) -> str:
    if not documents:
        return '''
        <div style="padding:60px 20px; text-align:center;">
            <div style="font-size:12px; color:#94a3b8; font-weight:500;">
                No extraction data
            </div>
            <div style="font-size:11px; color:#cbd5e1; margin-top:4px;">
                Upload documents above and click Activate to begin
            </div>
        </div>
        '''

    conf_styles = {
        "HIGH":     ("#ecfdf5", "#047857", "#059669"),
        "MEDIUM":   ("#eff6ff", "#1e40af", "#2563eb"),
        "LOW":      ("#fffbeb", "#92400e", "#d97706"),
        "VERY_LOW": ("#fef2f2", "#991b1b", "#dc2626"),
    }

    html = ['<div style="display:flex; flex-direction:column; gap:22px;">']

    for dt in DOC_TYPES:
        if dt not in documents:
            continue

        doc = documents[dt]
        html.append(f'''
        <div>
            <div style="display:flex; align-items:center; gap:10px; margin-bottom:10px;">
                <span style="font-size:11px; font-weight:700; color:#334155;
                             letter-spacing:1px; text-transform:uppercase;">
                    {DOC_LABELS[dt]}
                </span>
                <span style="flex:1; height:1px; background:#e5e7eb;"></span>
                <span style="font-size:10px; color:#94a3b8; font-weight:600;
                             font-family:'JetBrains Mono', monospace;">
                    {len(REQUIRED_FIELDS_BY_DOC.get(dt, []))} fields
                </span>
            </div>
            <div style="display:grid; grid-template-columns:repeat(auto-fill, minmax(220px, 1fr)); gap:8px;">
        ''')

        for field in REQUIRED_FIELDS_BY_DOC.get(dt, []):
            info = doc["extracted_fields"].get(field, {})
            value = info.get("value", "") or "—"
            conf = info.get("confidence", "VERY_LOW")
            bg, fg, dot = conf_styles.get(conf, ("#f8fafc", "#64748b", "#94a3b8"))

            html.append(f'''
            <div style="background:#ffffff; border:1px solid #e5e7eb; border-radius:8px;
                        padding:12px 14px; transition:border-color 0.1s;"
                 onmouseover="this.style.borderColor='#cbd5e1'"
                 onmouseout="this.style.borderColor='#e5e7eb'">
                <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
                    <span style="font-size:9px; color:#64748b; letter-spacing:0.8px;
                                text-transform:uppercase; font-weight:700;">
                        {FIELD_LABELS[field]}
                    </span>
                    <span style="display:inline-flex; align-items:center; gap:4px;
                                padding:2px 6px; border-radius:4px;
                                background:{bg}; color:{fg};
                                font-size:9px; font-weight:700; letter-spacing:0.5px;">
                        <span style="width:5px; height:5px; border-radius:50%; background:{dot};"></span>
                        {conf}
                    </span>
                </div>
                <div style="font-size:13px; font-weight:600; color:#0f172a;
                            word-break:break-word; line-height:1.4; min-height:18px;">
                    {value}
                </div>
            </div>
            ''')

        html.append('</div></div>')

    html.append('</div>')
    return "".join(html)


def empty_extraction_table() -> pd.DataFrame:
    return pd.DataFrame(columns=["Document", "Field", "Value", "Confidence"])


def build_editable_table(documents) -> pd.DataFrame:
    rows = []
    for dt in DOC_TYPES:
        if dt not in documents:
            continue
        for f in REQUIRED_FIELDS_BY_DOC.get(dt, []):
            info = documents[dt]["extracted_fields"].get(f, {})
            rows.append({
                "Document": DOC_LABELS[dt],
                "Field": FIELD_LABELS[f],
                "Value": info.get("value", ""),
                "Confidence": info.get("confidence", "VERY_LOW"),
            })
    return pd.DataFrame(rows)


def build_cross_check_html(cc_results) -> str:
    if not cc_results:
        return '<div style="padding:24px; text-align:center; color:#94a3b8; font-size:11px;">No cross-checkable fields</div>'

    items = []
    for it in cc_results:
        ok = it["status"] == "MATCH"
        color = "#059669" if ok else "#dc2626"
        icon = "✓" if ok else "✗"
        comparisons = "   ".join(
            f'<span style="color:#64748b; font-size:11px;">{DOC_LABELS.get(c["document"], c["document"])}</span> '
            f'<span style="color:#0f172a; font-size:12px; font-weight:600; font-family:\'JetBrains Mono\', monospace;">{c["value"]}</span>'
            for c in it["comparisons"]
        )

        items.append(f'''
        <div style="padding:11px 14px; background:#ffffff; border:1px solid #e5e7eb;
                    border-left:3px solid {color}; border-radius:6px; margin-bottom:6px;">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
                <span style="display:inline-flex; align-items:center; gap:6px;
                            font-weight:600; color:#0f172a; font-size:12px;">
                    <span style="color:{color}; font-weight:700;">{icon}</span>
                    {FIELD_LABELS.get(it["field"], it["field"])}
                </span>
                <span style="font-size:9px; color:#94a3b8; letter-spacing:0.8px;
                            background:#f8fafc; padding:2px 7px; border-radius:4px;
                            font-weight:700; font-family:'JetBrains Mono', monospace;">
                    {it["match_type"].upper()}
                </span>
            </div>
            <div style="display:flex; gap:14px; flex-wrap:wrap;">
                {comparisons}
            </div>
        </div>
        ''')
    return "".join(items)


def build_evidence_html(evidence) -> str:
    if not evidence:
        return '''
        <div style="padding:20px; background:#ffffff; border:1px solid #e5e7eb;
                    border-left:3px solid #059669; border-radius:6px;
                    font-size:12px; color:#059669; font-weight:600;">
            No risk flags raised
        </div>
        '''

    items = []
    for e in evidence:
        items.append(f'''
        <div style="padding:11px 14px; background:#ffffff; border:1px solid #e5e7eb;
                    border-left:3px solid #dc2626; border-radius:6px; margin-bottom:6px;">
            <div style="font-weight:600; color:#0f172a; font-size:12px; margin-bottom:6px;">
                <span style="color:#dc2626; margin-right:6px;">⚠</span>
                {e["risk_flag"]}
            </div>
            <div style="font-size:11px; color:#475569;">
                <span style="background:#f8fafc; padding:2px 8px; border-radius:4px;
                            font-family:'JetBrains Mono', monospace; font-size:11px;
                            color:#334155; border:1px solid #e5e7eb; font-weight:500;">
                    {FIELD_LABELS.get(e["field"], e["field"])} = {e["value"]}
                </span>
            </div>
        </div>
        ''')
    return "".join(items)


def build_fallback_banner(used_fallback: bool, extraction_errors: list) -> str:
    if not extraction_errors:
        return ""

    n = len(extraction_errors)
    lines = []
    for e in extraction_errors[:3]:
        lines.append(f"<li style='margin-bottom:2px;'>{str(e)[:200]}</li>")

    all_failed = all(
        "Failed after" in str(e) or "quota" in str(e).lower()
        or "429" in str(e) or "503" in str(e) or "404" in str(e)
        for e in extraction_errors
    )
    has_quota = any("429" in str(e) or "quota" in str(e).lower()
                    for e in extraction_errors)
    has_404 = any("404" in str(e) or "not found" in str(e).lower()
                  for e in extraction_errors)
    has_auth = any("401" in str(e) or "403" in str(e)
                   or "API_KEY" in str(e).upper()
                   for e in extraction_errors)

    if has_404:
        headline = "Model không tồn tại hoặc không có quyền truy cập"
        hint = ("Kiểm tra lại ô **Model** ở trên. Chọn 1 trong 4 preset: "
                "`gemini-3.6-flash`, `gemini-3.5-flash`, `gemini-3.5-flash-lite`, "
                "`gemini-3.1-pro-preview`. Bấm **Test API** để xác nhận.")
    elif has_auth:
        headline = "API Key không hợp lệ"
        hint = "Kiểm tra lại Gemini API Key. Bấm **Test API** để xác nhận."
    elif has_quota:
        headline = "Hết quota API"
        hint = "Đợi 1-2 phút rồi thử lại, hoặc dùng model khác (`gemini-3.5-flash-lite`)."
    elif all_failed:
        headline = f"AI extraction thất bại cho tất cả {n} tài liệu"
        hint = ("Nguyên nhân có thể: ảnh không đọc được, API lỗi tạm thời, "
                "hoặc JSON response không hợp lệ. Bấm **Test API** để kiểm tra.")
    else:
        headline = f"AI extraction thất bại cho {n} tài liệu"
        hint = "Một số trường không trích xuất được. Xem chi tiết bên dưới."

    return f'''
    <div style="padding:14px 18px; background:#fef2f2; border:1px solid #fecaca;
                border-left:4px solid #dc2626; border-radius:8px; margin-bottom:14px;
                font-size:12px; color:#7f1d1d;">
        <div style="font-weight:700; font-size:13px; margin-bottom:6px;">
            ⚠️ {headline}
        </div>
        <div style="margin-bottom:8px; color:#991b1b;">{hint}</div>
        <details style="margin-top:6px;">
            <summary style="cursor:pointer; font-weight:600; font-size:11px;">
                Chi tiết lỗi ({n})
            </summary>
            <ul style="margin:6px 0 0 20px; font-size:11px; line-height:1.5;">
                {"".join(lines)}
            </ul>
        </details>
    </div>
    '''


def build_pipeline_badge(done: bool = False) -> str:
    steps = ["Extract", "Confidence", "Cross-check", "Decision", "Review"]
    parts = []
    for i, name in enumerate(steps):
        if done:
            color, bg, border = "#059669", "#ecfdf5", "#a7f3d0"
            icon = "✓"
        else:
            color, bg, border = "#94a3b8", "#f8fafc", "#e5e7eb"
            icon = "○"
        parts.append(f'''
        <span style="display:inline-flex; align-items:center; gap:5px;
                     padding:5px 11px; background:{bg}; border:1px solid {border};
                     border-radius:20px; font-size:11px; font-weight:600; color:{color};">
            <span style="font-weight:700; font-size:10px;">{icon}</span> {name}
        </span>
        ''')
        if i < len(steps) - 1:
            parts.append('<span style="color:#cbd5e1; font-size:11px; margin:0 4px;">→</span>')

    return f'''
    <div style="display:flex; flex-wrap:wrap; align-items:center; justify-content:center;
                gap:4px; padding:12px 16px; background:#ffffff;
                border:1px solid #e5e7eb; border-radius:8px; margin-bottom:14px;">
        {"".join(parts)}
    </div>
    '''


def build_baseline_chart_html(df):
    bars = []
    for _, row in df.iterrows():
        acc_str = str(row["Accuracy"]).replace("%", "")
        try:
            acc = float(acc_str)
        except ValueError:
            acc = 0
        color = "#059669" if acc >= 90 else ("#2563eb" if acc >= 60 else "#94a3b8")
        bars.append(f'''
        <div style="margin-bottom:14px;">
            <div style="display:flex; justify-content:space-between; margin-bottom:5px;">
                <span style="font-weight:600; color:#0f172a; font-size:12px;">
                    {row["Baseline"]}
                </span>
                <span style="font-weight:700; color:{color}; font-size:12px;
                             font-family:'JetBrains Mono', monospace;">
                    {acc}%
                </span>
            </div>
            <div style="height:8px; background:#f1f5f9; border-radius:4px; overflow:hidden;">
                <div style="height:100%; width:{acc}%; background:{color};
                            transition:width 0.4s ease;"></div>
            </div>
        </div>
        ''')
    return f'<div style="padding:20px; background:#ffffff; border:1px solid #e5e7eb; border-radius:8px;">{"".join(bars)}</div>'


# =========================================================================
# PIPELINE
# =========================================================================

def process_uploaded_case(api_key, id_path, kyc_path, poa_path,
                          model: str = DEFAULT_MODEL, progress=None):
    start = time.perf_counter()
    case_id = f"CASE-{datetime.now().strftime('%Y%m%d-%H%M%S')}"
    model = _resolve_model(model)

    if progress is not None:
        progress(0.05, desc=f"⚡ Initializing Gemini client ({model})...")

    documents, errors = build_documents_from_uploads(api_key, id_path, kyc_path,
                                                     poa_path, model=model,
                                                     progress=progress)

    if progress is not None:
        progress(0.65, desc="⚙️ Running completeness check...")
    completeness = completeness_check(documents)

    if progress is not None:
        progress(0.72, desc="⚙️ Running cross-check...")
    cc_results = cross_check(documents)

    if progress is not None:
        progress(0.78, desc="⚙️ Checking document expiry...")
    expiry_result = expiry_check(documents)

    if progress is not None:
        progress(0.82, desc="⚙️ Computing confidence...")
    confidence_result = confidence_check(documents)

    if progress is not None:
        progress(0.86, desc="⚙️ Running decision engine...")
    decision = decision_engine(completeness, cc_results, expiry_result,
                                confidence_result, extraction_errors=errors)

    risk_flags, evidence = build_risk_flags_and_evidence(
        documents, cc_results, expiry_result, confidence_result
    )

    if progress is not None:
        progress(0.92, desc=f"🤖 Generating KYC Summary ({model})...")

    if decision == "SYSTEM_ERROR":
        summary_data = {
            "text": _build_offline_summary(decision, risk_flags, completeness),
            "used_fallback": True,
        }
    else:
        summary_data = generate_llm_summary_via_gemini(api_key, decision, risk_flags,
                                                        completeness, model=model)

    if progress is not None:
        progress(1.00, desc="✓ Done")

    return {
        "case_id": case_id, "documents": documents,
        "completeness_check": completeness, "cross_check_results": cc_results,
        "expiry_check": expiry_result,
        "confidence_check": confidence_result, "risk_flags": risk_flags,
        "evidence": evidence, "decision": decision,
        "llm_summary": summary_data["text"],
        "used_fallback": summary_data["used_fallback"],
        "extraction_errors": errors,
        "processing_time": round(time.perf_counter() - start, 2),
        "human_reviewed": False, "human_decision": None, "review_notes": "",
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "model": model,
    }


def reevaluate_case_with_edited_data(current_result, edited_df, reviewer_notes=""):
    if current_result is None:
        return None
    documents = _rebuild_documents_from_table(edited_df, current_result["documents"])
    completeness = completeness_check(documents)
    cc_results = cross_check(documents)
    expiry_result = expiry_check(documents)
    confidence_result = confidence_check(documents)
    decision = decision_engine(completeness, cc_results, expiry_result,
                                confidence_result, extraction_errors=None)
    risk_flags, evidence = build_risk_flags_and_evidence(
        documents, cc_results, expiry_result, confidence_result
    )
    updated = dict(current_result)
    updated.update({
        "documents": documents, "completeness_check": completeness,
        "cross_check_results": cc_results, "expiry_check": expiry_result,
        "confidence_check": confidence_result,
        "risk_flags": risk_flags, "evidence": evidence, "decision": decision,
        "review_notes": reviewer_notes or current_result.get("review_notes", ""),
    })
    return updated


def _rebuild_documents_from_table(df, original_documents):
    documents = {}
    doc_label_to_type = {v: k for k, v in DOC_LABELS.items()}
    field_label_to_key = {v: k for k, v in FIELD_LABELS.items()}
    for _, row in df.iterrows():
        doc_type = doc_label_to_type.get(str(row.get("Document", "")).strip())
        field = field_label_to_key.get(str(row.get("Field", "")).strip())
        if not doc_type or not field:
            continue
        value = row.get("Value", "")
        if value is None or (isinstance(value, float) and pd.isna(value)):
            value = ""
        else:
            value = str(value).strip()
        if doc_type not in documents:
            orig = original_documents.get(doc_type, {})
            documents[doc_type] = {
                "doc_type": doc_type, "file_name": orig.get("file_name", "edited"),
                "extracted_fields": {},
            }
        documents[doc_type]["extracted_fields"][field] = {
            "value": value, "confidence": calculate_field_confidence(field, value),
        }
    return documents


# =========================================================================
# SYNTHETIC TEST CASES
# =========================================================================

def _field(value, confidence="HIGH"):
    return {"value": value, "confidence": confidence}


def _base_documents():
    return {
        "ID_CARD": {"doc_type": "ID_CARD", "file_name": "sample_ID.png", "extracted_fields": {
            "full_name": _field("Nguyen Van An"), "dob": _field("1998-03-15"),
            "id_number": _field("ID100001"), "gender": _field("Male"),
            "expiry_date": _field("2030-03-15"),
        }},
        "KYC_FORM": {"doc_type": "KYC_FORM", "file_name": "sample_KYC.png", "extracted_fields": {
            "full_name": _field("Nguyen Van An"), "dob": _field("1998-03-15"),
            "id_number": _field("ID100001"), "gender": _field("Male"),
            "address": _field("123 Nguyen Trai, District 1, HCMC"),
        }},
        "PROOF_OF_ADDRESS": {"doc_type": "PROOF_OF_ADDRESS", "file_name": "sample_POA.png",
                              "extracted_fields": {
            "full_name": _field("Nguyen Van An"),
            "address": _field("123 Nguyen Trai, District 1, HCMC"),
        }},
    }


import copy as _copy


def _clone_with(base, doc_type, field, **kwargs):
    docs = _copy.deepcopy(base)
    docs[doc_type]["extracted_fields"][field] = _field(**kwargs)
    return docs


GROUND_TRUTH_TC01 = {
    "full_name": "Nguyen Van An", "dob": "1998-03-15", "id_number": "ID100001",
    "gender": "Male", "address": "123 Nguyen Trai, District 1, HCMC", "expiry_date": "2030-03-15",
}

_base = _base_documents()
SYNTHETIC_TEST_CASES = {
    "TC01": {"label": "TC01 — Clean", "documents": _base, "expected_decision": "ACCEPT"},
    "TC02": {"label": "TC02 — Name mismatch", "documents": _clone_with(_base, "KYC_FORM", "full_name", value="Tran Thi Lan"), "expected_decision": "MANUAL_REVIEW"},
    "TC03": {"label": "TC03 — ID mismatch", "documents": _clone_with(_base, "KYC_FORM", "id_number", value="ID100099"), "expected_decision": "MANUAL_REVIEW"},
    "TC04": {"label": "TC04 — DOB mismatch", "documents": _clone_with(_base, "KYC_FORM", "dob", value="1998-03-20"), "expected_decision": "MANUAL_REVIEW"},
    "TC05": {"label": "TC05 — Address mismatch", "documents": _clone_with(_base, "PROOF_OF_ADDRESS", "address", value="456 Le Loi, D3, HCMC"), "expected_decision": "MANUAL_REVIEW"},
    "TC08": {"label": "TC08 — Expired ID", "documents": _clone_with(_base, "ID_CARD", "expiry_date", value="2023-01-01"), "expected_decision": "MANUAL_REVIEW"},
    "TC09": {"label": "TC09 — Low confidence", "documents": _clone_with(_base, "ID_CARD", "full_name", value="Nguyen Van An", confidence="LOW"), "expected_decision": "MANUAL_REVIEW"},
}
_tc06 = _copy.deepcopy(_base); del _tc06["PROOF_OF_ADDRESS"]
SYNTHETIC_TEST_CASES["TC06"] = {"label": "TC06 — Missing POA", "documents": _tc06, "expected_decision": "REQUEST_MORE_INFO"}
_tc07 = _copy.deepcopy(_base); del _tc07["KYC_FORM"]["extracted_fields"]["address"]
SYNTHETIC_TEST_CASES["TC07"] = {"label": "TC07 — Missing field", "documents": _tc07, "expected_decision": "REQUEST_MORE_INFO"}
_tc10 = _copy.deepcopy(_base); _tc10["KYC_FORM"]["extracted_fields"]["id_number"] = _field("ID1O0O01", confidence="LOW")
SYNTHETIC_TEST_CASES["TC10"] = {"label": "TC10 — Extraction error", "documents": _tc10, "expected_decision": "MANUAL_REVIEW"}


# =========================================================================
# EVALUATION
# =========================================================================

def calculate_extraction_accuracy(documents, ground_truth):
    per_field = {}
    total, correct = 0, 0
    for field, expected_val in ground_truth.items():
        actual_val = None
        for dt in ["KYC_FORM", "ID_CARD", "PROOF_OF_ADDRESS"]:
            if dt in documents and field in documents[dt]["extracted_fields"]:
                actual_val = documents[dt]["extracted_fields"][field]["value"]
                break
        is_match = normalize_text(expected_val) == normalize_text(actual_val)
        per_field[field] = {"expected": expected_val, "actual": actual_val, "match": is_match}
        total += 1
        correct += int(is_match)
    return {"total": total, "correct": correct,
            "accuracy": round(correct / total * 100, 1) if total else 0, "per_field": per_field}


def calculate_hallucination_proxy(documents):
    total_fields = 0
    low_conf = 0
    for doc in documents.values():
        for info in doc["extracted_fields"].values():
            total_fields += 1
            if info.get("confidence") in ("LOW", "VERY_LOW"):
                low_conf += 1
    rate = round(low_conf / total_fields * 100, 1) if total_fields else 0
    return {"total_fields": total_fields, "low_confidence": low_conf, "rate": rate}


def calculate_completeness_detection():
    correct, total = 0, 0
    for tc_id in ["TC06"]:
        total += 1
        docs = _copy.deepcopy(SYNTHETIC_TEST_CASES[tc_id]["documents"])
        if completeness_check(docs)["missing_documents"]:
            correct += 1
    for tc_id in ["TC07"]:
        total += 1
        docs = _copy.deepcopy(SYNTHETIC_TEST_CASES[tc_id]["documents"])
        if completeness_check(docs)["missing_fields"]:
            correct += 1
    return {"correct": correct, "total": total,
            "accuracy": round(correct / total * 100, 1) if total else 0}


def run_full_evaluation():
    rows = []
    correct_decision = 0
    total_low_conf = 0
    total_fields = 0

    for cid, meta in SYNTHETIC_TEST_CASES.items():
        docs = _copy.deepcopy(meta["documents"])
        sys_dec = decision_engine(
            completeness_check(docs), cross_check(docs), expiry_check(docs),
            confidence_check(docs),
        )
        exp_dec = meta["expected_decision"]
        dec_ok = exp_dec == sys_dec
        correct_decision += int(dec_ok)
        h = calculate_hallucination_proxy(docs)
        total_low_conf += h["low_confidence"]
        total_fields += h["total_fields"]
        rows.append({
            "Case": cid, "Description": meta["label"],
            "Expected": exp_dec, "System": sys_dec,
            "OK": "✓" if dec_ok else "✗",
        })

    df = pd.DataFrame(rows)
    n = len(SYNTHETIC_TEST_CASES)
    decision_acc = round(correct_decision / n * 100, 1)
    halluc_rate = round(total_low_conf / total_fields * 100, 1) if total_fields else 0
    extract_acc = calculate_extraction_accuracy(
        _copy.deepcopy(SYNTHETIC_TEST_CASES["TC01"]["documents"]), GROUND_TRUTH_TC01,
    )
    completeness_det = calculate_completeness_detection()

    summary = f"""
### Evaluation Results · {n} Test Cases

| # | Metric | Value | Target | Status |
|---|---|---|---|---|
| 1 | **Decision Accuracy** | {correct_decision}/{n} = **{decision_acc}%** | ≥ 90% | {"✓" if decision_acc >= 90 else "!"} |
| 2 | **Extraction Accuracy** | {extract_acc['correct']}/{extract_acc['total']} = **{extract_acc['accuracy']}%** | ≥ 80% | {"✓" if extract_acc['accuracy'] >= 80 else "!"} |
| 3 | **Hallucination Rate** | {total_low_conf}/{total_fields} = **{halluc_rate}%** | ≤ 15% | {"✓" if halluc_rate <= 15 else "!"} |
| 4 | **Completeness Detection** | {completeness_det['correct']}/{completeness_det['total']} = **{completeness_det['accuracy']}%** | 100% | {"✓" if completeness_det['accuracy'] == 100 else "!"} |
"""
    return df, summary, extract_acc["per_field"]


def compare_baselines():
    baselines = {
        "LLM-only (no Rule Engine)": 0,
        "Rule Engine (completeness only)": 0,
        "Full System": 0,
    }
    n = len(SYNTHETIC_TEST_CASES)
    for cid, meta in SYNTHETIC_TEST_CASES.items():
        docs = _copy.deepcopy(meta["documents"])
        exp = meta["expected_decision"]
        comp = completeness_check(docs)
        llm_dec = "REQUEST_MORE_INFO" if (comp["missing_documents"] or comp["missing_fields"]) else "ACCEPT"
        baselines["LLM-only (no Rule Engine)"] += int(llm_dec == exp)
        re_dec = decision_engine(comp, [], {"expired": None}, {"overall": "HIGH", "low_confidence_fields": []})
        baselines["Rule Engine (completeness only)"] += int(re_dec == exp)
        full_dec = decision_engine(comp, cross_check(docs), expiry_check(docs), confidence_check(docs))
        baselines["Full System"] += int(full_dec == exp)
    return pd.DataFrame([
        {"Baseline": k, "Correct": f"{v}/{n}", "Accuracy": f"{round(v/n*100, 1)}%"}
        for k, v in baselines.items()
    ])


def analyze_failure_cases(df):
    failures = df[df["OK"] == "✗"]
    if failures.empty:
        return "### ✓ No Failure Cases\n\nAll test cases returned the expected decision."
    lines = ["### ✗ Failure Cases Analysis", ""]
    for _, row in failures.iterrows():
        lines.append(f"**{row['Case']}** — {row['Description']}")
        lines.append(f"- Expected: `{row['Expected']}` | System: `{row['System']}`")
        lines.append("")
    return "\n".join(lines)


# =========================================================================
# EXCEL EXPORT
# =========================================================================

def export_audit_report(result: dict) -> str:
    if result is None:
        return None
    wb = Workbook()
    hf = Font(name="Arial", bold=True, color="FFFFFF")
    hfill = PatternFill(start_color="0F172A", end_color="0F172A", fill_type="solid")
    bf = Font(name="Arial")

    def _h(ws, row):
        for c in ws[row]:
            c.font = hf; c.fill = hfill
            c.alignment = Alignment(horizontal="left", vertical="center")

    def _b(ws, start=2):
        for row in ws.iter_rows(min_row=start):
            for c in row: c.font = bf

    ws1 = wb.active; ws1.title = "Summary"
    ws1.append(["Case ID", "Created", "Model", "Decision", "Confidence",
                "Processing (s)", "Human Reviewed", "Human Decision", "Review Notes"])
    _h(ws1, 1)
    ws1.append([result["case_id"], result.get("created_at", ""),
                result.get("model", DEFAULT_MODEL), result["decision"],
                result["confidence_check"]["overall"],
                result["processing_time"], "Yes" if result["human_reviewed"] else "No",
                result["human_decision"] or "-", result.get("review_notes", "")])
    _b(ws1)

    ws2 = wb.create_sheet("Extracted Fields")
    ws2.append(["Document", "Field", "Value", "Confidence"]); _h(ws2, 1)
    for dt, doc in result["documents"].items():
        for f, info in doc["extracted_fields"].items():
            ws2.append([DOC_LABELS.get(dt, dt), FIELD_LABELS.get(f, f), info["value"], info["confidence"]])
    _b(ws2)

    ws3 = wb.create_sheet("Cross-check")
    ws3.append(["Field", "Status", "Match Type", "Comparisons"]); _h(ws3, 1)
    for it in result["cross_check_results"]:
        comp = " | ".join(f"{DOC_LABELS.get(c['document'], c['document'])}: {c['value']}" for c in it["comparisons"])
        ws3.append([FIELD_LABELS.get(it["field"], it["field"]), it["status"], it["match_type"], comp])
    _b(ws3)

    ws4 = wb.create_sheet("Risk Flags & Evidence")
    ws4.append(["Risk Flag Type", "Severity", "Message"]); _h(ws4, 1)
    for flag in result["risk_flags"]:
        ws4.append([flag["type"], flag["severity"], flag["message"]])
    ev_row = ws4.max_row + 2
    ws4.append([]); ws4.append(["Evidence — Risk Flag", "Document", "Field", "Value"])
    _h(ws4, ev_row + 1)
    for ev in result["evidence"]:
        ws4.append([ev["risk_flag"], DOC_LABELS.get(ev["document"], ev["document"]),
                    FIELD_LABELS.get(ev["field"], ev["field"]), ev["value"]])
    _b(ws4)

    for ws in wb.worksheets:
        for col in ws.columns:
            length = max((len(str(c.value)) if c.value else 0) for c in col)
            ws.column_dimensions[col[0].column_letter].width = min(max(length + 2, 12), 50)

    tmp = tempfile.mkdtemp()
    fp = os.path.join(tmp, f"KYC_Audit_Report_{result['case_id']}.xlsx")
    wb.save(fp)
    return fp


# =========================================================================
# CALLBACKS
# =========================================================================

def on_test_api(api_key, model):
    return test_gemini_api(api_key, model)


def on_process(api_key, model, id_img, kyc_img, poa_img, history, progress=gr.Progress()):
    key = (api_key or "").strip() or os.getenv("GEMINI_API_KEY", "").strip()
    if not key:
        raise gr.Error("Missing Gemini API Key. Please enter it above or set GEMINI_API_KEY in .env")

    selected_model = _resolve_model(model)
    print(f"[INFO] Processing with model: {selected_model}")

    if not any([id_img, kyc_img, poa_img]):
        raise gr.Error("Please upload at least one document.")

    for img, label in [(id_img, "ID Card"), (kyc_img, "KYC Form"), (poa_img, "Proof of Address")]:
        if img and not _is_valid_image(img):
            raise gr.Error(f"{label}: file must be an image (PNG / JPG / WEBP).")

    progress(0.02, desc=f"🚀 Starting analysis with {selected_model}...")
    result = process_uploaded_case(key, id_img, kyc_img, poa_img,
                                   model=selected_model, progress=progress)
    history = history + [result]

    return (
        build_sidebar_html(history, result),
        build_topbar_html(result),
        build_pipeline_badge(done=True),
        build_fallback_banner(result.get("used_fallback", False), result["extraction_errors"]),
        build_decision_banner(result["decision"], len(result["risk_flags"]), result["processing_time"]),
        build_field_cards(result["documents"]),
        build_editable_table(result["documents"]),
        build_cross_check_html(result["cross_check_results"]),
        build_evidence_html(result["evidence"]),
        result["llm_summary"],
        history,
        result,
        [],
    )


def on_reevaluate(edited_df, current_result, history, reviewer_notes=""):
    if current_result is None:
        raise gr.Error("No case to re-evaluate. Please run analysis first.")
    updated = reevaluate_case_with_edited_data(current_result, edited_df, reviewer_notes)
    new_history = []
    for r in history:
        if r["case_id"] == updated["case_id"]:
            new_history.append(updated)
        else:
            new_history.append(r)
    gr.Info(f"♻️ Re-evaluated: Decision is now {updated['decision']}")
    return (
        build_sidebar_html(new_history, updated),
        build_topbar_html(updated),
        build_decision_banner(updated["decision"], len(updated["risk_flags"]), updated["processing_time"]),
        build_field_cards(updated["documents"]),
        build_editable_table(updated["documents"]),
        build_cross_check_html(updated["cross_check_results"]),
        build_evidence_html(updated["evidence"]),
        updated["llm_summary"],
        new_history,
        updated,
    )


def on_quick_review(choice_label, notes, history, current_result):
    if current_result is None:
        gr.Warning("No case to review.")
        return build_sidebar_html(history, current_result)
    current_result["human_reviewed"] = True
    current_result["human_decision"] = choice_label
    current_result["review_notes"] = notes or ""
    new_history = []
    for r in history:
        if r["case_id"] == current_result["case_id"]:
            new_history.append(current_result)
        else:
            new_history.append(r)
    gr.Info(f"✓ Recorded: {choice_label}")
    return build_sidebar_html(new_history, current_result)


def on_reset_session():
    gr.Info("🔄 Session reset")
    return (
        [],
        build_sidebar_html([]),
        build_topbar_html(),
        build_decision_banner("", 0),
        build_field_cards({}),
        empty_extraction_table(),
        build_cross_check_html([]),
        build_evidence_html([]),
        "_Run analysis to generate summary._",
        None,
        [{"role": "assistant", "content": "Hello! Ask me anything about this case. I can explain the decision, cite evidence, or clarify cross-check results."}],
    )


def on_ai_assistant_ask(api_key, model, question, current_result, chat_history):
    effective_model = (current_result or {}).get("model") or model
    return ask_ai_assistant(api_key, question, current_result, chat_history,
                            model=effective_model)


def on_export(current_result):
    if current_result is None:
        raise gr.Error("No case to export.")
    return export_audit_report(current_result)


def on_run_evaluation():
    df, summary, extract_per_field = run_full_evaluation()
    failure_analysis = analyze_failure_cases(df)
    extract_df = pd.DataFrame([
        {"Field": FIELD_LABELS.get(f, f), "Expected": v["expected"],
         "Actual": v["actual"], "Match?": "✓" if v["match"] else "✗"}
        for f, v in extract_per_field.items()
    ])
    return df, summary + "\n\n" + failure_analysis, extract_df


def on_run_baseline():
    df = compare_baselines()
    return df, build_baseline_chart_html(df)


# =========================================================================
# HEAD HTML — BULLETPROOF SIDEBAR (position: fixed)
# =========================================================================

HEAD_HTML = """
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700;800&family=JetBrains+Mono:wght@400;500;600&display=swap" rel="stylesheet">

<style>
  * { font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important; }
  code, pre, .gr-code { font-family: 'JetBrains Mono', monospace !important; }

  footer { display: none !important; }
  .gradio-container { max-width: 100% !important; padding: 0 !important; margin: 0 !important; }
  .gradio-container > .main { padding: 0 !important; gap: 0 !important; }
  .gradio-container .wrap { padding: 0 !important; }
  .gradio-container .gr-block { border: none !important; background: transparent !important; padding: 0 !important; }
  .gradio-container .gr-form { border: none !important; background: transparent !important; }
  .gradio-container .gr-panel { border: none !important; background: transparent !important; }
  .gradio-container .gr-box { border: none !important; background: transparent !important; }
  .gradio-container .gr-group { border: none !important; background: transparent !important; padding: 0 !important; gap: 0 !important; }

  html, body, .gradio-container {
    background: #f8fafc !important;
    color: #0f172a !important;
    min-height: 100vh;
    overflow-anchor: none !important;
  }

  .gradio-container, .gradio-container p, .gradio-container span,
  .gradio-container label, .gradio-container h1, .gradio-container h2,
  .gradio-container h3, .gradio-container h4, .gradio-container td,
  .gradio-container th, .gradio-container li, .gradio-container div {
    color: #0f172a;
  }

  .layout-row {
    display: block !important;
    padding: 0 0 0 260px !important;
    margin: 0 !important;
    min-height: 100vh !important;
    gap: 0 !important;
    overflow: visible !important;
  }

  .sidebar-column,
  .sidebar-column > div,
  .sidebar-column > .gr-html,
  .sidebar-column > .html-container {
    position: fixed !important;
    left: 0 !important;
    top: 0 !important;
    bottom: 0 !important;
    width: 260px !important;
    min-width: 260px !important;
    max-width: 260px !important;
    height: 100vh !important;
    max-height: 100vh !important;
    padding: 0 !important;
    margin: 0 !important;
    background: #ffffff !important;
    border-right: 1px solid #e5e7eb !important;
    overflow: hidden !important;
    z-index: 100 !important;
    box-sizing: border-box !important;
    display: block !important;
  }

  .sidebar-column .prose,
  .sidebar-column .md,
  .sidebar-column .html-container {
    padding: 0 !important;
    margin: 0 !important;
    max-width: 100% !important;
    height: 100% !important;
  }

  #sidebar-html,
  #sidebar-html > div,
  #sidebar-html .html-container,
  #sidebar-html .prose {
    padding: 0 !important;
    margin: 0 !important;
    height: 100% !important;
    min-height: 0 !important;
    max-width: 100% !important;
    overflow: hidden !important;
  }

  #kyc-sidebar {
    height: 100% !important;
    min-height: 100vh !important;
  }

  .main-column {
    width: 100% !important;
    max-width: 100% !important;
    padding: 0 !important;
    margin: 0 !important;
    background: #f8fafc !important;
  }

  .kyc-reset-html,
  #reset-session-btn {
    all: unset !important;
    box-sizing: border-box !important;
    display: block !important;
    width: 100% !important;
    padding: 16px 20px !important;
    background: #fafbfc !important;
    color: #64748b !important;
    border: none !important;
    border-top: 1px solid #e5e7eb !important;
    border-radius: 0 !important;
    font-family: 'Inter', -apple-system, BlinkMacSystemFont, sans-serif !important;
    font-size: 11px !important;
    font-weight: 600 !important;
    letter-spacing: 0.5px !important;
    text-align: left !important;
    line-height: 1.4 !important;
    cursor: pointer !important;
    transition: all 0.15s ease !important;
    outline: none !important;
    user-select: none !important;
    -webkit-tap-highlight-color: transparent !important;
    box-shadow: none !important;
    text-decoration: none !important;
  }
  .kyc-reset-html:hover,
  #reset-session-btn:hover {
    background: #fef2f2 !important;
    color: #dc2626 !important;
    border-top-color: #fecaca !important;
  }
  .kyc-reset-html:focus,
  #reset-session-btn:focus {
    outline: none !important;
    box-shadow: none !important;
  }

  #hidden-reset-btn {
    position: absolute !important;
    left: -9999px !important;
    top: -9999px !important;
    width: 1px !important;
    height: 1px !important;
    overflow: hidden !important;
    opacity: 0 !important;
    pointer-events: none !important;
  }

  #sidebar-html, #sidebar-html *,
  #topbar-html, #topbar-html *,
  #decision-box, #decision-box *,
  #pipeline-badge, #pipeline-badge *,
  #field-cards, #field-cards *,
  #cross-check-html, #cross-check-html *,
  #evidence-html, #evidence-html * {
    color: unset;
  }

  * { overflow-anchor: none !important; }
  html { scroll-padding-top: 200px !important; }
  .tabitem {
    min-height: 400px !important;
    scroll-margin-top: 300px !important;
    overflow-anchor: none !important;
  }
  [role="tabpanel"] {
    min-height: 400px !important;
    scroll-margin-top: 300px !important;
    overflow-anchor: none !important;
  }
  button[role="tab"] { outline: none !important; outline-offset: 0 !important; }
  button[role="tab"]:focus,
  button[role="tab"]:focus-visible { outline: none !important; box-shadow: none !important; }
  .tab-nav, [role="tablist"] { overflow-anchor: none !important; scroll-margin-top: 0 !important; }
  .main-content-padding, .main-column { overflow-anchor: none !important; scroll-margin-top: 0 !important; }

  .image-container [role="switch"],
  .image-frame [role="switch"],
  [data-testid="image"] [role="switch"],
  .image-container input[type="checkbox"],
  .image-frame input[type="checkbox"],
  [data-testid="image"] input[type="checkbox"],
  .image-container .toggle,
  .image-frame .toggle,
  [data-testid="image"] .toggle {
    display: none !important;
    visibility: hidden !important;
    opacity: 0 !important;
    pointer-events: none !important;
    width: 0 !important;
    height: 0 !important;
    position: absolute !important;
    left: -9999px !important;
  }

  #extraction-table,
  #extraction-table > div,
  #extraction-table .table-wrap,
  #extraction-table .tabulator {
    min-height: 0 !important;
  }
  #extraction-table .tabulator-tableholder {
    max-height: 320px !important;
    overflow-y: auto !important;
  }

  .clean-card {
    background: #ffffff !important;
    border-radius: 8px !important;
    padding: 20px 24px !important;
    box-shadow: 0 1px 2px rgba(15,23,42,0.03) !important;
    border: 1px solid #e5e7eb !important;
    margin-bottom: 14px !important;
  }

  .tabs { border: none !important; }
  button[role="tab"] {
    font-weight: 500 !important; font-size: 13px !important;
    padding: 10px 18px !important; color: #64748b !important;
    border: none !important; border-radius: 0 !important;
    background: transparent !important;
    border-bottom: 1px solid transparent !important;
    transition: color 0.1s;
    letter-spacing: -0.1px;
  }
  button[role="tab"]:hover { color: #0f172a !important; }
  button[role="tab"][aria-selected="true"] {
    color: #0f172a !important;
    border-bottom: 1px solid #0f172a !important;
    background: transparent !important;
    font-weight: 600 !important;
  }
  .result-tabs button[role="tab"] {
    font-size: 12px !important; padding: 8px 14px !important;
  }

  button.primary, .primary button, button[variant="primary"] {
    background: #0f172a !important;
    border: none !important;
    font-weight: 600 !important;
    letter-spacing: -0.1px;
    box-shadow: 0 1px 2px rgba(15,23,42,0.1) !important;
    transition: background 0.1s !important;
    color: #ffffff !important;
    border-radius: 6px !important;
    font-size: 13px !important;
  }
  button.primary:hover, .primary button:hover { background: #1e293b !important; }
  button.secondary, .secondary button {
    background: #ffffff !important; color: #334155 !important;
    border: 1px solid #e5e7eb !important; font-weight: 500 !important;
    transition: all 0.1s !important;
    border-radius: 6px !important;
    font-size: 13px !important;
  }
  button.secondary:hover, .secondary button:hover {
    border-color: #cbd5e1 !important;
    background: #f8fafc !important;
  }

  table { font-size: 12px !important; border-collapse: collapse !important; }
  thead th {
    background: #f8fafc !important;
    color: #475569 !important;
    font-weight: 600 !important; font-size: 10px !important;
    letter-spacing: 0.8px; text-transform: uppercase;
    padding: 10px 14px !important;
    border-bottom: 1px solid #e5e7eb !important;
  }
  tbody td {
    padding: 10px 14px !important;
    border-bottom: 1px solid #f1f5f9 !important;
    font-size: 12px !important;
  }

  details {
    border: 1px solid #e5e7eb !important; border-radius: 6px !important;
    margin-bottom: 8px !important;
    background: #ffffff !important;
    overflow: hidden;
  }
  details summary {
    padding: 12px 16px !important; font-weight: 600 !important;
    color: #334155 !important; cursor: pointer; font-size: 12px !important;
    transition: background 0.1s;
    letter-spacing: -0.1px;
  }
  details summary:hover { background: #f8fafc; }
  details[open] summary { border-bottom: 1px solid #e5e7eb; }

  input, textarea, select {
    font-size: 13px !important; border-radius: 6px !important;
    border-color: #e5e7eb !important;
    transition: border-color 0.1s !important;
    background: #ffffff !important;
  }
  input:focus, textarea:focus, select:focus {
    border-color: #0f172a !important;
    box-shadow: 0 0 0 3px rgba(15,23,42,0.06) !important;
    outline: none !important;
  }

  ::-webkit-scrollbar { width: 8px; height: 8px; }
  ::-webkit-scrollbar-track { background: transparent; }
  ::-webkit-scrollbar-thumb {
    background: rgba(148,163,184,0.4);
    border-radius: 4px;
  }
  ::-webkit-scrollbar-thumb:hover { background: rgba(100,116,139,0.6); }

  .main-content-padding { padding: 22px 32px 40px !important; }

  .section-marker { display: flex; align-items: center; gap: 12px; margin-bottom: 16px; }
  .section-num {
    width: 24px; height: 24px; border-radius: 6px;
    background: #f1f5f9;
    display: flex; align-items: center; justify-content: center;
    color: #475569; font-size: 11px; font-weight: 700;
    flex-shrink: 0;
    border: 1px solid #e5e7eb;
  }
  .section-title {
    font-size: 13px; font-weight: 600; color: #0f172a;
    letter-spacing: -0.2px;
  }
  .section-subtitle { font-size: 11px; color: #64748b; margin-top: 2px; }

  .image-container, .image-frame, [data-testid="image"] {
    position: relative !important;
  }
  .image-container .icon-buttons,
  .image-frame .icon-buttons,
  [data-testid="image"] .icon-buttons,
  .image-container [class*="icon-button"],
  .image-frame [class*="icon-button"] {
    position: absolute !important;
    top: 8px !important;
    right: 8px !important;
    bottom: auto !important;
    left: auto !important;
    z-index: 100 !important;
    display: flex !important;
    flex-direction: row !important;
    gap: 2px !important;
    background: rgba(255,255,255,0.96) !important;
    backdrop-filter: blur(8px) !important;
    -webkit-backdrop-filter: blur(8px) !important;
    border-radius: 6px !important;
    padding: 3px 5px !important;
    border: 1px solid #e5e7eb !important;
    box-shadow: 0 1px 3px rgba(15,23,42,0.08) !important;
  }
  .image-container .icon-buttons button,
  .image-frame .icon-buttons button {
    background: transparent !important;
    border: none !important;
    padding: 4px !important;
    border-radius: 4px !important;
    color: #475569 !important;
    transition: background 0.1s !important;
  }
  .image-container .icon-buttons button:hover,
  .image-frame .icon-buttons button:hover {
    background: #f1f5f9 !important;
    color: #0f172a !important;
  }
</style>

<script>
(function () {
  function stripDark() {
    document.documentElement.classList.remove('dark');
    document.body.classList.remove('dark');
  }
  stripDark();
  var obs = new MutationObserver(function () { stripDark(); });
  obs.observe(document.documentElement, { attributes: true, attributeFilter: ['class'] });
  obs.observe(document.body, { attributes: true, attributeFilter: ['class'] });

  var scrollLockActive = false;
  var lockedY = 0;
  var lockedX = 0;

  function lockScroll() {
    lockedY = window.scrollY;
    lockedX = window.scrollX;
    scrollLockActive = true;
    setTimeout(function () { scrollLockActive = false; }, 800);
  }

  function restoreScroll() {
    if (!scrollLockActive) return;
    if (Math.abs(window.scrollY - lockedY) > 1 || Math.abs(window.scrollX - lockedX) > 1) {
      window.scrollTo({ top: lockedY, left: lockedX, behavior: 'instant' });
    }
  }

  document.addEventListener('click', function (e) {
    var tab = e.target.closest('button[role="tab"]');
    if (!tab) return;
    lockScroll();
  }, true);

  document.addEventListener('focusin', function (e) {
    if (e.target.closest('button[role="tab"]')) lockScroll();
  }, true);

  document.addEventListener('scroll', function (e) {
    if (scrollLockActive) restoreScroll();
  }, true);

  function tick() {
    if (scrollLockActive) restoreScroll();
    requestAnimationFrame(tick);
  }
  requestAnimationFrame(tick);

  function hideMirrorToggle() {
    var selectors = [
      '.image-container [role="switch"]',
      '.image-frame [role="switch"]',
      '[data-testid="image"] [role="switch"]',
      '.image-container input[type="checkbox"]',
      '.image-frame input[type="checkbox"]',
      '[data-testid="image"] input[type="checkbox"]'
    ];
    selectors.forEach(function (sel) {
      document.querySelectorAll(sel).forEach(function (el) {
        el.style.display = 'none';
        el.style.visibility = 'hidden';
        el.style.opacity = '0';
        el.style.pointerEvents = 'none';
        el.style.position = 'absolute';
        el.style.left = '-9999px';
        el.setAttribute('aria-hidden', 'true');
      });
    });
  }
  setInterval(hideMirrorToggle, 200);
  window.addEventListener('load', hideMirrorToggle);

  function moveImageButtons() {
    var containers = document.querySelectorAll('.image-container, .image-frame, [data-testid="image"]');
    containers.forEach(function (container) {
      if (container.dataset.kycPatched) return;
      var children = container.querySelectorAll(':scope > div');
      children.forEach(function (child) {
        var buttons = child.querySelectorAll(':scope > button');
        if (buttons.length >= 2) {
          var hasIcon = false;
          buttons.forEach(function (b) {
            if (b.querySelector('svg') || (b.getAttribute('aria-label') || '').match(/upload|camera|clipboard|download/i)) {
              hasIcon = true;
            }
          });
          if (hasIcon) {
            child.style.position = 'absolute';
            child.style.top = '8px';
            child.style.right = '8px';
            child.style.bottom = 'auto';
            child.style.left = 'auto';
            child.style.zIndex = '100';
            child.style.background = 'rgba(255,255,255,0.96)';
            child.style.backdropFilter = 'blur(8px)';
            child.style.borderRadius = '6px';
            child.style.padding = '3px 5px';
            child.style.border = '1px solid #e5e7eb';
            child.style.boxShadow = '0 1px 3px rgba(15,23,42,0.08)';
            child.style.display = 'flex';
            child.style.flexDirection = 'row';
            child.style.gap = '2px';
          }
        }
      });
      container.dataset.kycPatched = 'true';
    });
  }
  setInterval(moveImageButtons, 400);
  window.addEventListener('load', moveImageButtons);
})();
</script>
"""


# =========================================================================
# GRADIO UI
# =========================================================================

WELCOME_MESSAGE = [{"role": "assistant", "content": "Hello! Ask me anything about this case. I can explain the decision, cite evidence, or clarify cross-check results."}]

with gr.Blocks(title="KYC Copilot") as demo:

    history_state = gr.State([])
    current_result_state = gr.State(None)

    with gr.Row(elem_classes="layout-row", equal_height=False):

        with gr.Column(scale=0, min_width=260, elem_classes="sidebar-column"):
            sidebar_html = gr.HTML(build_sidebar_html([]), elem_id="sidebar-html")

        with gr.Column(scale=1, elem_classes="main-column"):

            topbar_html = gr.HTML(build_topbar_html(), elem_id="topbar-html")

            with gr.Column(elem_classes="main-content-padding"):

                hidden_reset_btn = gr.Button("reset", elem_id="hidden-reset-btn", visible=False)

                with gr.Tabs():

                    with gr.Tab("Onboarding"):

                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('''
                            <div class="section-marker">
                                <div class="section-num">1</div>
                                <div>
                                    <div class="section-title">Upload Documents</div>
                                    <div class="section-subtitle">Provide ID Card, KYC Form, and Proof of Address</div>
                                </div>
                            </div>
                            ''')
                            with gr.Row():
                                api_key_box = gr.Textbox(
                                    label="Gemini API Key",
                                    type="password",
                                    placeholder="Nhập API key (hoặc để trống nếu có GEMINI_API_KEY trong .env)",
                                    scale=3,
                                )
                                model_dropdown = gr.Dropdown(
                                    label="Model",
                                    choices=MODEL_PRESETS,
                                    value=DEFAULT_MODEL,
                                    allow_custom_value=True,
                                    interactive=True,
                                    info="Chọn preset hoặc gõ tên model bất kỳ",
                                    scale=2,
                                )
                            with gr.Row():
                                test_api_btn = gr.Button("🔍 Test API Key + Model", variant="secondary", size="sm")
                            test_api_result = gr.Markdown("")
                            with gr.Row():
                                id_image = gr.Image(type="filepath", label="ID Card / Passport", height=150)
                                kyc_image = gr.Image(type="filepath", label="KYC Form", height=150)
                                poa_image = gr.Image(type="filepath", label="Proof of Address", height=150)
                            process_btn = gr.Button("Activate AI Analysis", variant="primary", size="lg")

                        pipeline_bar = gr.HTML(build_pipeline_badge(done=False), elem_id="pipeline-badge")
                        fallback_banner = gr.HTML()
                        decision_box = gr.HTML(build_decision_banner("", 0), elem_id="decision-box")

                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('''
                            <div class="section-marker">
                                <div class="section-num">2</div>
                                <div>
                                    <div class="section-title">Analysis Results</div>
                                    <div class="section-subtitle">Extracted fields, cross-check, and evidence</div>
                                </div>
                            </div>
                            ''')
                            with gr.Tabs(elem_classes="result-tabs"):
                                with gr.Tab("Fields"):
                                    field_cards = gr.HTML(build_field_cards({}), elem_id="field-cards")
                                with gr.Tab("Cross-check"):
                                    cross_check_html = gr.HTML(build_cross_check_html([]), elem_id="cross-check-html")
                                with gr.Tab("Evidence"):
                                    evidence_html = gr.HTML(build_evidence_html([]), elem_id="evidence-html")
                                with gr.Tab("Summary"):
                                    with gr.Row():
                                        with gr.Column(scale=5):
                                            summary_box = gr.Markdown("_Run analysis to generate summary._")
                                        with gr.Column(scale=1, min_width=100):
                                            copy_summary_btn = gr.Button("📋 Copy", variant="secondary", size="sm")

                            with gr.Accordion("✎ Edit Values — correct AI mistakes, then Re-evaluate below", open=False):
                                extraction_table = gr.Dataframe(
                                    value=empty_extraction_table(),
                                    headers=["Document", "Field", "Value", "Confidence"],
                                    datatype=["str", "str", "str", "str"],
                                    interactive=True,
                                    wrap=True,
                                    elem_id="extraction-table",
                                )

                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('''
                            <div class="section-marker">
                                <div class="section-num">3</div>
                                <div>
                                    <div class="section-title">Human Review</div>
                                    <div class="section-subtitle">Reviewer approves before any action is taken</div>
                                </div>
                            </div>
                            ''')
                            reviewer_notes_box = gr.Textbox(
                                label="Review Notes",
                                placeholder="Optional — reason for change",
                            )
                            with gr.Row():
                                reeval_btn = gr.Button("↻ Re-evaluate Rules", variant="secondary")
                                accept_btn = gr.Button("Accept", variant="primary")
                                request_btn = gr.Button("Request Info")
                                manual_btn = gr.Button("Manual Review")
                            with gr.Row():
                                export_btn = gr.Button("Export Audit Report (.xlsx)")
                            export_file = gr.File(label="", interactive=False, show_label=False)

                        with gr.Accordion("AI Assistant — Ask about this case", open=False):
                            chatbot = gr.Chatbot(
                                value=WELCOME_MESSAGE,
                                height=220,
                                show_label=False,
                            )
                            with gr.Row():
                                chat_input = gr.Textbox(placeholder="e.g. Why was this case flagged?", scale=4, show_label=False)
                                chat_send = gr.Button("Send", variant="primary", scale=1)

                    with gr.Tab("Evaluation"):
                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('''
                            <div class="section-marker">
                                <div class="section-num">·</div>
                                <div>
                                    <div class="section-title">Full Evaluation</div>
                                    <div class="section-subtitle">10 test cases · 4 metrics · no AI calls</div>
                                </div>
                            </div>
                            ''')
                            eval_btn = gr.Button("Run Evaluation", variant="primary")
                            eval_summary = gr.Markdown()
                            eval_table = gr.Dataframe(interactive=False)

                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('''
                            <div class="section-marker">
                                <div class="section-num">·</div>
                                <div>
                                    <div class="section-title">Baseline Comparison</div>
                                    <div class="section-subtitle">Demonstrates the value of the Rule Engine</div>
                                </div>
                            </div>
                            ''')
                            baseline_btn = gr.Button("Compare Baselines", variant="secondary")
                            baseline_chart = gr.HTML()
                            baseline_table = gr.Dataframe(interactive=False)

                        with gr.Group(elem_classes=["clean-card"]):
                            gr.HTML('<div class="section-title" style="margin-bottom:14px;">Extraction Accuracy Detail</div>')
                            extract_acc_table = gr.Dataframe(interactive=False)

    test_api_btn.click(
        fn=on_test_api,
        inputs=[api_key_box, model_dropdown],
        outputs=[test_api_result],
    )

    process_btn.click(
        fn=on_process,
        inputs=[api_key_box, model_dropdown, id_image, kyc_image, poa_image, history_state],
        outputs=[
            sidebar_html, topbar_html,
            pipeline_bar, fallback_banner, decision_box,
            field_cards, extraction_table, cross_check_html,
            evidence_html, summary_box,
            history_state, current_result_state, chatbot,
        ],
    )

    reeval_btn.click(
        fn=on_reevaluate,
        inputs=[extraction_table, current_result_state, history_state, reviewer_notes_box],
        outputs=[
            sidebar_html, topbar_html, decision_box,
            field_cards, extraction_table, cross_check_html,
            evidence_html, summary_box,
            history_state, current_result_state,
        ],
    )

    accept_btn.click(
        fn=lambda notes, h, cur: on_quick_review("Accept", notes, h, cur),
        inputs=[reviewer_notes_box, history_state, current_result_state],
        outputs=[sidebar_html],
    )
    request_btn.click(
        fn=lambda notes, h, cur: on_quick_review("Request More Information", notes, h, cur),
        inputs=[reviewer_notes_box, history_state, current_result_state],
        outputs=[sidebar_html],
    )
    manual_btn.click(
        fn=lambda notes, h, cur: on_quick_review("Manual Review", notes, h, cur),
        inputs=[reviewer_notes_box, history_state, current_result_state],
        outputs=[sidebar_html],
    )

    export_btn.click(fn=on_export, inputs=[current_result_state], outputs=[export_file])

    hidden_reset_btn.click(
        fn=on_reset_session,
        inputs=[],
        outputs=[
            history_state, sidebar_html, topbar_html, decision_box,
            field_cards, extraction_table, cross_check_html,
            evidence_html, summary_box, current_result_state, chatbot,
        ],
    )

    copy_summary_btn.click(
        fn=None,
        inputs=[summary_box],
        outputs=[],
        js="(text) => { if (text) navigator.clipboard.writeText(text); return []; }",
    )

    chat_send.click(
        fn=on_ai_assistant_ask,
        inputs=[api_key_box, model_dropdown, chat_input, current_result_state, chatbot],
        outputs=[chatbot, chat_input],
    )
    chat_input.submit(
        fn=on_ai_assistant_ask,
        inputs=[api_key_box, model_dropdown, chat_input, current_result_state, chatbot],
        outputs=[chatbot, chat_input],
    )

    eval_btn.click(fn=on_run_evaluation, inputs=[], outputs=[eval_table, eval_summary, extract_acc_table])
    baseline_btn.click(fn=on_run_baseline, inputs=[], outputs=[baseline_table, baseline_chart])


if __name__ == "__main__":
    demo.launch(
        head=HEAD_HTML,
        theme=gr.themes.Base(primary_hue="slate", neutral_hue="slate"),
        # share=True,
    )



It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f98a4c06f8a0e15f0c.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
